# Lab 2A: Streaming and vision
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 2 · about 30 minutes**

**Objectives**
- Measure time to first token with and without streaming
- Read the raw stream events in order
- Build a stream helper that checks the final message
- Send images with text and validate what comes back

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic matplotlib
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## Part 1: time to first token
**Predict:** will streaming change the total time? Will it change when the first words appear?

In [ ]:
QUESTION = "Explain, in about 250 words, how a fiber broadband connection gets from the exchange to a home router."

t0 = time.perf_counter()
full = client.messages.create(model=config.MODEL, max_tokens=800, messages=[{"role": "user", "content": QUESTION}])
t_block = time.perf_counter() - t0
print(f"Non-streamed: first words appear after {t_block:.2f}s (the whole answer at once)")

t0, first = time.perf_counter(), None
with client.messages.stream(model=config.MODEL, max_tokens=800, messages=[{"role": "user", "content": QUESTION}]) as stream:
    for text in stream.text_stream:
        if first is None:
            first = time.perf_counter() - t0
        print(text, end="", flush=True)
final = stream.get_final_message()
t_stream = time.perf_counter() - t0
print(f"\n\nStreamed: first token after {first:.2f}s, finished after {t_stream:.2f}s")
print("final stop_reason:", final.stop_reason, "| usage:", final.usage)

## Part 2: what arrives on the wire
Iterating the stream itself (not just `text_stream`) shows every event.

In [ ]:
from collections import Counter
order, counts = [], Counter()
with client.messages.stream(model=config.MODEL, max_tokens=200,
                            messages=[{"role": "user", "content": "Give three tips for a stable home WiFi signal."}]) as stream:
    for event in stream:
        counts[event.type] += 1
        if not order or order[-1] != event.type:
            order.append(event.type)
print("event order (repeats collapsed):")
print("  " + " -> ".join(order))
print("counts:", dict(counts))

You will see raw events such as `message_start`, `content_block_start`, `content_block_delta`, `content_block_stop`, `message_delta`, and `message_stop`, plus SDK convenience events such as `text`.
The final `stop_reason` and output usage arrive in `message_delta`.

## Part 3: a safe stream helper
A stream can end early (network drop or a mid-stream error event). Never show partial text as a finished answer.

In [ ]:
def stream_answer(messages, max_tokens=800, on_text=lambda t: print(t, end="", flush=True), **kw):
    """Stream text to the UI; return (final_message, complete_flag). Incomplete answers are flagged, never silently kept."""
    try:
        with client.messages.stream(model=config.MODEL, max_tokens=max_tokens, messages=messages, **kw) as stream:
            for text in stream.text_stream:
                on_text(text)
            final = stream.get_final_message()
    except (anthropic.APIStatusError, anthropic.APIConnectionError) as e:
        print(f"\n[stream failed: {type(e).__name__}] Show a clear error or retry with backoff.")
        return None, False
    complete = final.stop_reason in ("end_turn", "stop_sequence")
    if not complete:
        print(f"\n[incomplete: stop_reason={final.stop_reason}]")
    return final, complete

final, ok = stream_answer([{"role": "user", "content": "List five causes of slow broadband, one line each."}])
print("\ncomplete:", ok, "| output_tokens:", final.usage.output_tokens if final else None)
final, ok = stream_answer([{"role": "user", "content": "Describe every layer of the OSI model in depth."}], max_tokens=40)
print("\ncomplete:", ok)

## Part 4: vision with a chart you generate
We draw a synthetic chart, so we know the true values and can check Claude's reading.

In [ ]:
import base64, io
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
MONTHS = ["Apr", "May", "Jun", "Jul", "Aug", "Sep"]
TICKETS = [412, 388, 455, 501, 476, 530]
fig, ax = plt.subplots(figsize=(6, 3.2), dpi=120)
bars = ax.bar(MONTHS, TICKETS, color="#2B4C7E")
ax.bar_label(bars); ax.set_title("Router support tickets per month"); ax.set_ylim(0, 600)
buf = io.BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig)
chart_b64 = base64.b64encode(buf.getvalue()).decode()
from IPython.display import Image as ShowImage, display
display(ShowImage(data=buf.getvalue()))

In [ ]:
schema_hint = 'Reply with only JSON: {"months": ["Apr", ...], "values": [412, ...]}'
resp = client.messages.create(model=config.MODEL, max_tokens=300, messages=[{"role": "user", "content": [
    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": chart_b64}},
    {"type": "text", "text": "Read every bar label in this chart. " + schema_hint}]}])
raw = text_of(resp).strip().removeprefix("```json").removesuffix("```").strip()
got = json.loads(raw)
print("Claude read:", got)
errors = [(m, t, v) for m, t, v in zip(MONTHS, TICKETS, got.get("values", [])) if t != v]
print("all values correct" if not errors and len(got.get("values", [])) == len(TICKETS) else f"mismatches: {errors}")
print("input_tokens for this image request:", resp.usage.input_tokens)

Validate numbers in code, as you just did. Try it: send the same data with `media_type` set to `image/jpeg` and read the error.

## Part 5: two labelled images in one prompt

In [ ]:
def chart_png(values, title):
    fig, ax = plt.subplots(figsize=(6, 3.2), dpi=120)
    b = ax.bar(MONTHS, values, color="#2B4C7E"); ax.bar_label(b); ax.set_title(title); ax.set_ylim(0, 600)
    out = io.BytesIO(); fig.savefig(out, format="png", bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(out.getvalue()).decode()
north = chart_png([300, 320, 310, 350, 360, 340], "Region North tickets")
south = chart_png([112, 68, 145, 151, 116, 190], "Region South tickets")
img = lambda d: {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": d}}
resp = client.messages.create(model=config.MODEL, max_tokens=400, messages=[{"role": "user", "content": [
    {"type": "text", "text": "Image 1:"}, img(north), {"type": "text", "text": "Image 2:"}, img(south),
    {"type": "text", "text": "In which month is the gap between Image 1 and Image 2 smallest? Name the month and both values."}]}])
print(text_of(resp))
gaps = {m: a - b for m, a, b in zip(MONTHS, [300, 320, 310, 350, 360, 340], [112, 68, 145, 151, 116, 190])}
print("\nground truth: smallest gap in", min(gaps, key=gaps.get), gaps)

## You should now have
- [ ] Time to first token measured with and without streaming
- [ ] `stream_answer()` that flags incomplete answers
- [ ] A vision request with values validated in code
- [ ] A labelled multi-image prompt

## Check yourself
1. What changes with streaming: total time, cost, or time to first token?
2. Where do the final stop_reason and usage arrive in a stream?
3. Why label images and put them before the question?